# Persistent homology with `tda`

One function does everything: `tda.persistent_homology(data)`. The input is a
point cloud (`(n, d)` array), a distance matrix, or a sparse distance graph —
everything else is inferred or has a sensible default.

```
pip install tda
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tda

rng = np.random.default_rng(0)

## A noisy circle

`barcode[d]` is a `(k, 2)` array of `[birth, death]` intervals for homology
dimension `d`. A circle should show one dominant H1 interval.

In [ ]:
theta = rng.uniform(0, 2 * np.pi, 300)
points = np.stack([np.cos(theta), np.sin(theta)], axis=1)
points += rng.normal(scale=0.08, size=points.shape)

barcode = tda.persistent_homology(points, max_dim=1)
print(f"H0: {len(barcode[0])} intervals   H1: {len(barcode[1])} intervals")

In [ ]:
def plot_diagram(barcode, ax=None):
    ax = ax or plt.gca()
    finite = np.concatenate([d[np.isfinite(d[:, 1])] for d in barcode])
    top = finite.max() * 1.05 if len(finite) else 1.0
    ax.plot([0, top], [0, top], 'k--', lw=0.5)
    for dim, dgm in enumerate(barcode):
        deaths = np.where(np.isfinite(dgm[:, 1]), dgm[:, 1], top)
        ax.scatter(dgm[:, 0], deaths, s=14, label=f'H{dim}')
    ax.set_xlabel('birth'); ax.set_ylabel('death'); ax.legend()

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 4))
ax0.scatter(*points.T, s=6); ax0.set_aspect('equal'); ax0.set_title('data')
plot_diagram(barcode, ax1); ax1.set_title('persistence diagram')
plt.tight_layout()

The point far from the diagonal is the circle. Everything near the diagonal
is noise.

## A sphere: homology beyond H1

`max_dim=2` finds the 2-dimensional void of a sphere sample.

In [ ]:
z = 1 - 2 * (np.arange(256) + 0.5) / 256
r = np.sqrt(1 - z * z)
golden = np.pi * (3 - np.sqrt(5))
sphere = np.stack([r * np.cos(golden * np.arange(256)),
                   r * np.sin(golden * np.arange(256)), z], axis=1)

barcode = tda.persistent_homology(sphere, max_dim=2)
h2 = barcode[2]
dominant = h2[np.argmax(h2[:, 1] - h2[:, 0])]
print(f"dominant H2 interval: [{dominant[0]:.3f}, {dominant[1]:.3f})")

## Other inputs

A precomputed distance matrix (`distance_matrix=True`), or any
`scipy.sparse` matrix as a distance graph: entries are edges, missing pairs
are never connected (works for disconnected data too).

In [ ]:
diff = points[:, None, :] - points[None, :, :]
distances = np.sqrt((diff ** 2).sum(-1))

from_points = tda.persistent_homology(points, max_dim=1)
from_matrix = tda.persistent_homology(distances, max_dim=1, distance_matrix=True)

print('identical H1:', np.array_equal(from_points[1], from_matrix[1]))

## Scaling up: the `quotient` approximation

`quotient=True` computes on a much smaller quotient-cover filtration —
a `log(3)`-interleaved approximation. Dominant features survive; runtime and
filtration size drop dramatically (this is the path to deep dimensions and
large point counts).

In [ ]:
import time

for quotient in (False, True):
    size = tda.filtration_size(sphere, max_dim=2, quotient=quotient)
    start = time.perf_counter()
    bc = tda.persistent_homology(sphere, max_dim=2, quotient=quotient)
    elapsed = time.perf_counter() - start
    h2 = bc[2]
    top = h2[np.argmax(h2[:, 1] - h2[:, 0])]
    label = 'quotient' if quotient else 'exact   '
    print(f"{label}  simplices: {size:>9,}   time: {elapsed:.3f}s   "
          f"dominant H2: [{top[0]:.3f}, {top[1]:.3f})")

That is the whole API: `persistent_homology` and `filtration_size`, with
`max_dim`, `threshold`, `distance_matrix`, `quotient`, `peel` (an exact
speedup, on by default), `collapse`, and `parallel` as the only knobs.
For maps into circles, tori, and projective spaces, see the coordinates
notebook.